# LeetCode 3859 — Count Subarrays With K Distinct Integers bằng Python

- Link: https://leetcode.com/problems/count-subarrays-with-k-distinct-integers/
- Độ khó: **Hard**
- Chủ đề: Cửa sổ trượt (two pointers), đếm "đúng k" = "ít nhất k" − "ít nhất k+1", tính đơn điệu của điều kiện trên đoạn, `Counter`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countSubarrays(self, nums: list[int], k: int, m: int) -> int:
```

## 1. Mục tiêu học

1. Hiểu điều kiện kép: đoạn có **đúng `k` giá trị phân biệt** và **mỗi giá trị xuất hiện ≥ `m` lần**.
2. Viết lại điều kiện bằng hai đại lượng đơn điệu theo phép **mở rộng đoạn**: `d` = số giá trị phân biệt, `t` = số giá trị có tần suất ≥ `m`. Điều kiện ⇔ `d = k` và `t ≥ k`.
3. Áp dụng mẹo **"đúng k = ít nhất k − ít nhất k+1"** để chỉ phải đếm các tính chất **đơn điệu**.
4. Cài đặt cửa sổ trượt đếm "số đầu trái hợp lệ cho mỗi đầu phải" trong `O(n)`.
5. Viết phiên bản **một lượt với hai con trỏ trái** và đối chiếu với vét cạn.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums` và hai số `k`, `m`. Đếm số đoạn con (liên tiếp, khác rỗng) thỏa:

- đoạn chứa **đúng `k`** số nguyên phân biệt;
- trong đoạn, **mỗi** số phân biệt xuất hiện **ít nhất `m` lần**.

**Ví dụ 1:** `nums = [1,2,1,2,2], k = 2, m = 2` → `2` (`[1,2,1,2]` và `[1,2,1,2,2]`).

**Ví dụ 2:** `nums = [3,1,2,4], k = 2, m = 1` → `3` (`[3,1]`, `[1,2]`, `[2,4]`).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^5`, `1 ≤ k, m ≤ n`.

In [1]:
from collections import Counter
import random, time

examples = [([1, 2, 1, 2, 2], 2, 2, 2), ([3, 1, 2, 4], 2, 1, 3)]
for nums, k, m, exp in examples:
    print(nums, "k =", k, "m =", m, "-> kỳ vọng", exp)

[1, 2, 1, 2, 2] k = 2 m = 2 -> kỳ vọng 2
[3, 1, 2, 4] k = 2 m = 1 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. `Counter` và theo dõi số giá trị phân biệt

`Counter` đếm tần suất. Khi tần suất về 0 phải **xóa khóa** (`pop`) để `len(cnt)` đúng bằng số giá trị phân biệt.

In [2]:
cnt = Counter([1, 2, 1])
print(cnt, len(cnt))
cnt[2] -= 1
print("chưa xóa:", len(cnt), "| ", end="")
if cnt[2] == 0:
    cnt.pop(2)
print("sau khi xóa:", len(cnt))

Counter({1: 2, 2: 1}) 2
chưa xóa: 2 | sau khi xóa: 1


### 3.2. Đếm "số giá trị đạt ngưỡng" khi tăng/giảm tần suất

Giữ biến `t` = số giá trị có tần suất `≥ m`. Khi một tần suất **tăng lên đúng `m`** thì `t += 1`; khi **giảm từ `m` xuống `m - 1`** thì `t -= 1`. Không cần duyệt lại `Counter`.

In [3]:
m = 2; cnt = Counter(); t = 0
for x in [5, 5, 7, 5, 7]:
    cnt[x] += 1
    if cnt[x] == m:
        t += 1
    print(f"thêm {x}: cnt={dict(cnt)} t={t}")

thêm 5: cnt={5: 1} t=0
thêm 5: cnt={5: 2} t=1
thêm 7: cnt={5: 2, 7: 1} t=1
thêm 5: cnt={5: 3, 7: 1} t=1
thêm 7: cnt={5: 3, 7: 2} t=2


### 3.3. Cửa sổ trượt đếm đoạn có tính chất "đơn điệu khi mở rộng"

Nếu tính chất `P` thỏa trên `[l, r]` thì cũng thỏa trên mọi đoạn **chứa** nó, thì với mỗi `r`, tập các `l` hợp lệ là một **tiền tố** `[0, L(r))` và `L(r)` không giảm khi `r` tăng. Mẫu: thêm `nums[r]`; **trong khi** `[l, r]` còn thỏa `P` thì bỏ `nums[l]`, `l += 1`; cuối cùng cộng `l` (số đầu trái hợp lệ `0..l-1`).

In [4]:
# Ví dụ: đếm đoạn có tổng >= S (mảng không âm) — tính chất đơn điệu khi mở rộng
def count_sum_at_least(a, S):
    l = cur = ans = 0
    for x in a:
        cur += x
        while cur >= S:
            cur -= a[l]; l += 1
        ans += l
    return ans

a = [2, 1, 3, 1]
print(count_sum_at_least(a, 4), sum(1 for i in range(4) for j in range(i, 4) if sum(a[i:j + 1]) >= 4))

5 5


### 3.4. "Đúng k" = "ít nhất k" − "ít nhất k+1"

"Đúng `k` giá trị phân biệt" **không** đơn điệu (mở rộng có thể làm vượt `k`). Nhưng "**ít nhất** `k`" thì đơn điệu. Vì vậy `#(d = k) = #(d ≥ k) − #(d ≥ k+1)` (kỹ thuật của LC 992 *Subarrays with K Different Integers*).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với một đoạn, đặt `d` = số giá trị phân biệt, `t` = số giá trị xuất hiện `≥ m` lần. Luôn có `t ≤ d`. Điều kiện đề bài:

`d = k` **và** mọi giá trị có tần suất `≥ m` ⇔ `d = k` **và** `t = k` ⇔ `d = k` **và** `t ≥ k`.

Khi **mở rộng** đoạn, cả `d` và `t` đều **không giảm**. Vậy:

- `P_k = (d ≥ k và t ≥ k)` đơn điệu;
- `P_{k+1}' = (d ≥ k+1 và t ≥ k)` đơn điệu;
- đoạn thỏa đề ⇔ `P_k` đúng nhưng `P_{k+1}'` sai.

Tính tay ví dụ 1, `[1,2,1,2,2]`, `k = m = 2`: các đoạn có `d = 2` và cả `1`, `2` xuất hiện ≥ 2 lần chỉ có `[1,2,1,2]` (chỉ số 0..3) và `[1,2,1,2,2]` (0..4). Đoạn `[2,1,2,2]` có `1` chỉ một lần ⇒ không.

In [5]:
def stats(seg, m):
    c = Counter(seg)
    return len(c), sum(1 for v in c.values() if v >= m)

for seg in ([1, 2, 1, 2], [2, 1, 2, 2], [1, 2, 1, 2, 2], [2, 2]):
    print(seg, "(d, t) =", stats(seg, 2))

[1, 2, 1, 2] (d, t) = (2, 2)
[2, 1, 2, 2] (d, t) = (2, 1)
[1, 2, 1, 2, 2] (d, t) = (2, 2)
[2, 2] (d, t) = (1, 1)


## 5. Cách 1 — Brute Force: kiểm tra mọi đoạn

Với mọi cặp `(l, r)`, dựng `Counter` và kiểm tra. `O(n³)` (hoặc `O(n²)` nếu cập nhật dần).

In [6]:
def brute(nums, k, m):
    n = len(nums); ans = 0
    for l in range(n):
        for r in range(l, n):
            c = Counter(nums[l:r + 1])
            if len(c) == k and all(v >= m for v in c.values()):
                ans += 1
    return ans

for nums, k, m, exp in examples:
    print(brute(nums, k, m), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


**Nút thắt:** số đoạn `n²/2 = 5·10^9`. Cần khai thác tính đơn điệu.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Quy về hai tính chất đơn điệu:** `answer = F(k) − F(k+1)` với

`F(lim) = #{đoạn : d ≥ lim và t ≥ k}`.

(`F(k)` đếm các đoạn có `d ≥ k, t ≥ k`; trừ đi những đoạn có `d ≥ k+1, t ≥ k` thì còn đúng các đoạn `d = k, t ≥ k`.)

**Bước 2 — Cửa sổ trượt cho `F(lim)`:** với mỗi đầu phải `r`, đẩy `l` sang phải **trong khi** `[l, r]` còn thỏa (`d ≥ lim` và `t ≥ k`). Khi dừng, mọi đầu trái `0..l-1` đều thỏa (vì đoạn dài hơn cũng thỏa) và `l` trở đi thì không ⇒ cộng `l`.

**Bước 3 (tùy chọn) — Một lượt:** với mỗi `r`, đầu trái hợp lệ là `l ∈ [lo(r), hi(r)]` với `lo(r)` = đầu trái nhỏ nhất mà `d ≤ k`, `hi(r)` = đầu trái lớn nhất mà `t ≥ k` và `d ≥ k`. Cả `lo`, `hi` không giảm theo `r` ⇒ hai con trỏ trái cùng chạy.

## 7. Các cách tiếp theo

- **Cách 2:** cố định `l`, mở rộng `r` và cập nhật `d`, `t` dần; dừng sớm khi `d > k` — `O(n²)` tệ nhất.
- **Cách 3:** `F(k) − F(k+1)` bằng hai lượt cửa sổ trượt — `O(n)` (lời giải nộp chính).
- **Cách 4:** một lượt với **hai con trỏ trái** `lo`, `hi` — `O(n)`.

### 7.1. Cách 2 — Mở rộng dần từ mỗi đầu trái

In [7]:
def expand(nums, k, m):
    n = len(nums); ans = 0
    for l in range(n):
        cnt = Counter(); t = 0
        for r in range(l, n):
            x = nums[r]
            cnt[x] += 1
            if cnt[x] == m:
                t += 1
            if len(cnt) > k:
                break                       # d chỉ tăng: không còn đoạn hợp lệ
            if len(cnt) == k and t == k:
                ans += 1
    return ans

for nums, k, m, exp in examples:
    print(expand(nums, k, m), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** `O(n²)` tệ nhất (ví dụ `k` lớn hoặc mảng ít giá trị phân biệt). Dừng sớm giúp nhiều khi `k` nhỏ và giá trị đa dạng.

### 7.2. Cách 3 — `F(k) − F(k+1)` (lời giải chính)

**Giải thích từng đoạn:**

1. `F(lim)`: `cnt`, `t`, `l`. Thêm `nums[r]`, cập nhật `t` khi tần suất chạm `m`.
2. `while len(cnt) >= lim and t >= k`: bỏ `nums[l]` — nếu tần suất từ `m` xuống `m-1` thì `t -= 1`; nếu về 0 thì xóa khóa; `l += 1`.
3. `ans += l`.
4. Trả `F(k) − F(k+1)`.

In [8]:
def at_least(nums, lim, k, m):
    cnt = Counter(); t = 0; l = 0; ans = 0
    for x in nums:
        cnt[x] += 1
        if cnt[x] == m:
            t += 1
        while len(cnt) >= lim and t >= k:
            y = nums[l]
            cnt[y] -= 1
            if cnt[y] == m - 1:
                t -= 1
            if cnt[y] == 0:
                del cnt[y]
            l += 1
        ans += l
    return ans

def two_pass(nums, k, m):
    return at_least(nums, k, k, m) - at_least(nums, k + 1, k, m)

for nums, k, m, exp in examples:
    print(two_pass(nums, k, m), "kỳ vọng", exp, "| F(k) =", at_least(nums, k, k, m), "F(k+1) =", at_least(nums, k + 1, k, m))

2 kỳ vọng 2 | F(k) = 2 F(k+1) = 0
3 kỳ vọng 3 | F(k) = 6 F(k+1) = 3


Lưu ý `m = 1`: khi thêm một giá trị mới, tần suất 1 = `m` ⇒ `t` tăng; khi xóa từ 1 về 0 thì `m - 1 = 0` ⇒ `t` giảm trước rồi xóa khóa — thứ tự hai lệnh `if` đúng.

**Độ phức tạp Cách 3:** mỗi phần tử vào/ra cửa sổ một lần mỗi lượt ⇒ `O(n)`; bộ nhớ `O(n)`.

### 7.3. Cách 4 — Một lượt với hai con trỏ trái

**Giải thích:**

- Cửa sổ A `[lo, r]`: giữ `d ≤ k` — đẩy `lo` khi số phân biệt vượt `k`. Mọi `l ≥ lo` có `d ≤ k`.
- Cửa sổ B `[hi, r]`: đẩy `hi` **trong khi** `[hi, r]` còn có `d ≥ k` và `t ≥ k` (giống `F(k)`). Sau vòng lặp, các `l < hi` thỏa `d ≥ k, t ≥ k`.
- Đầu trái hợp lệ: `lo ≤ l < hi` ⇒ cộng `max(0, hi − lo)`.

In [9]:
def one_pass(nums, k, m):
    ca = Counter(); lo = 0            # cửa sổ A: d <= k
    cb = Counter(); tb = 0; hi = 0    # cửa sổ B: như F(k)
    ans = 0
    for x in nums:
        ca[x] += 1
        while len(ca) > k:
            y = nums[lo]; ca[y] -= 1
            if ca[y] == 0: del ca[y]
            lo += 1
        cb[x] += 1
        if cb[x] == m: tb += 1
        while len(cb) >= k and tb >= k:
            y = nums[hi]; cb[y] -= 1
            if cb[y] == m - 1: tb -= 1
            if cb[y] == 0: del cb[y]
            hi += 1
        ans += max(0, hi - lo)
    return ans

for nums, k, m, exp in examples:
    print(one_pass(nums, k, m), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


In [10]:
class SolutionOnePass:
    def countSubarrays(self, nums: list[int], k: int, m: int) -> int:
        return one_pass(nums, k, m)

print([SolutionOnePass().countSubarrays(a, k, m) for a, k, m, _ in examples])

[2, 3]


**Độ phức tạp Cách 4:** `O(n)` một lượt (hai cửa sổ), bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace `F(lim)` của Cách 3 trên ví dụ 1 (`[1,2,1,2,2]`, `k = m = 2`).

`F(2)` (`d ≥ 2` và `t ≥ 2`):

| r | x | cnt sau khi thêm | t | đẩy l? | l | cộng |
|---|---|---|---|---|---|---|
| 0 | 1 | {1:1} | 0 | không | 0 | 0 |
| 1 | 2 | {1:1, 2:1} | 0 | không | 0 | 0 |
| 2 | 1 | {1:2, 2:1} | 1 | không | 0 | 0 |
| 3 | 2 | {1:2, 2:2} | 2 | bỏ `1` → t = 1, dừng | 1 | 1 |
| 4 | 2 | {1:1, 2:3} | 1 | không | 1 | 1 |

`F(2) = 2`. `F(3)` cần `d ≥ 3` — không bao giờ ⇒ `0`. Đáp án `2 − 0 = 2` ✔.

In [11]:
def trace_F(nums, lim, k, m):
    cnt = Counter(); t = 0; l = 0; ans = 0
    print(f"--- F(lim={lim}) ---")
    for r, x in enumerate(nums):
        cnt[x] += 1
        if cnt[x] == m: t += 1
        removed = []
        while len(cnt) >= lim and t >= k:
            y = nums[l]; cnt[y] -= 1
            if cnt[y] == m - 1: t -= 1
            if cnt[y] == 0: del cnt[y]
            removed.append(y); l += 1
        ans += l
        print(f"r={r} x={x} cnt={dict(cnt)} t={t} bỏ={removed} l={l} -> cộng {l}, tổng={ans}")
    return ans

a = trace_F([1, 2, 1, 2, 2], 2, 2, 2)
b = trace_F([1, 2, 1, 2, 2], 3, 2, 2)
print("đáp án =", a - b)

--- F(lim=2) ---
r=0 x=1 cnt={1: 1} t=0 bỏ=[] l=0 -> cộng 0, tổng=0
r=1 x=2 cnt={1: 1, 2: 1} t=0 bỏ=[] l=0 -> cộng 0, tổng=0
r=2 x=1 cnt={1: 2, 2: 1} t=1 bỏ=[] l=0 -> cộng 0, tổng=0
r=3 x=2 cnt={1: 1, 2: 2} t=1 bỏ=[1] l=1 -> cộng 1, tổng=1
r=4 x=2 cnt={1: 1, 2: 3} t=1 bỏ=[] l=1 -> cộng 1, tổng=2
--- F(lim=3) ---
r=0 x=1 cnt={1: 1} t=0 bỏ=[] l=0 -> cộng 0, tổng=0
r=1 x=2 cnt={1: 1, 2: 1} t=0 bỏ=[] l=0 -> cộng 0, tổng=0
r=2 x=1 cnt={1: 2, 2: 1} t=1 bỏ=[] l=0 -> cộng 0, tổng=0
r=3 x=2 cnt={1: 2, 2: 2} t=2 bỏ=[] l=0 -> cộng 0, tổng=0
r=4 x=2 cnt={1: 2, 2: 3} t=2 bỏ=[] l=0 -> cộng 0, tổng=0
đáp án = 2


In [12]:
# Trace ví dụ 2 (m = 1): F(2) - F(3)
a = trace_F([3, 1, 2, 4], 2, 2, 1)
b = trace_F([3, 1, 2, 4], 3, 2, 1)
print("đáp án =", a - b)

--- F(lim=2) ---
r=0 x=3 cnt={3: 1} t=1 bỏ=[] l=0 -> cộng 0, tổng=0
r=1 x=1 cnt={1: 1} t=1 bỏ=[3] l=1 -> cộng 1, tổng=1
r=2 x=2 cnt={2: 1} t=1 bỏ=[1] l=2 -> cộng 2, tổng=3
r=3 x=4 cnt={4: 1} t=1 bỏ=[2] l=3 -> cộng 3, tổng=6
--- F(lim=3) ---
r=0 x=3 cnt={3: 1} t=1 bỏ=[] l=0 -> cộng 0, tổng=0
r=1 x=1 cnt={3: 1, 1: 1} t=2 bỏ=[] l=0 -> cộng 0, tổng=0
r=2 x=2 cnt={1: 1, 2: 1} t=2 bỏ=[3] l=1 -> cộng 1, tổng=1
r=3 x=4 cnt={2: 1, 4: 1} t=2 bỏ=[1] l=2 -> cộng 2, tổng=3
đáp án = 3


## 9. Code nộp LeetCode

In [13]:
from collections import Counter

class Solution:
    def countSubarrays(self, nums: list[int], k: int, m: int) -> int:
        def F(lim):
            cnt = Counter()
            t = l = ans = 0
            for x in nums:
                cnt[x] += 1
                if cnt[x] == m:
                    t += 1
                while len(cnt) >= lim and t >= k:
                    y = nums[l]
                    cnt[y] -= 1
                    if cnt[y] == m - 1:
                        t -= 1
                    if cnt[y] == 0:
                        del cnt[y]
                    l += 1
                ans += l
            return ans

        return F(k) - F(k + 1)

sol = Solution()
for nums, k, m, exp in examples:
    got = sol.countSubarrays(nums, k, m)
    print(nums, k, m, "->", got, "✔" if got == exp else "✘")

[1, 2, 1, 2, 2] 2 2 -> 2 ✔
[3, 1, 2, 4] 2 1 -> 3 ✔


## 10. Vì sao thuật toán đúng

1. **Quy đổi điều kiện:** mỗi giá trị xuất hiện ≥ `m` lần ⇔ `t = d`. Kết hợp `d = k` ⇔ `d = k` và `t ≥ k` (vì `t ≤ d`).
2. **Hiệu hai tập:** `{d ≥ k, t ≥ k} \ {d ≥ k+1, t ≥ k} = {d = k, t ≥ k}` và tập sau là tập con của tập trước ⇒ `#` = `F(k) − F(k+1)`.
3. **Đơn điệu:** thêm phần tử vào đoạn không làm giảm `d` hay `t` (tần suất chỉ tăng) ⇒ với tính chất `P = (d ≥ lim và t ≥ k)`, nếu `[l, r]` thỏa thì `[l', r]` với `l' < l` cũng thỏa. Vậy với mỗi `r`, các `l` hợp lệ là `[0, L(r))`.
4. **Bất biến cửa sổ:** sau vòng `while`, `[l, r]` **không** thỏa `P` nhưng `[l-1, r]` thỏa (nếu `l > 0`) ⇒ `L(r) = l`. `L(r)` không giảm theo `r` (đoạn `[l, r+1] ⊇ [l, r]`) nên con trỏ không cần lùi.
5. Cập nhật `t` đúng: `t` thay đổi chỉ khi một tần suất **vượt qua** ngưỡng `m` (lên tới `m` hoặc xuống `m-1`).

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n³)` | `O(n)` |
| 2. Mở rộng từ mỗi `l` | `O(n²)` tệ nhất | `O(n)` |
| 3. `F(k) − F(k+1)` | `O(n)` (2 lượt) | `O(n)` |
| 4. Một lượt hai con trỏ trái | `O(n)` | `O(n)` |

Đáp án có thể tới `n(n+1)/2 ≈ 5·10^9` (ví dụ `k = 1`, `m = 1`, mảng toàn một giá trị) — vượt `int32` ở ngôn ngữ khác.

## 12. Bộ test

In [14]:
def run_all(nums, k, m, with_brute=True):
    res = [expand(nums, k, m), two_pass(nums, k, m), one_pass(nums, k, m), Solution().countSubarrays(nums, k, m)]
    if with_brute:
        res.append(brute(nums, k, m))
    return res

for nums, k, m, exp in examples:
    assert all(v == exp for v in run_all(nums, k, m))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [15]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([7], 1, 1, 1),
    ([7], 1, 2, 0),                    # m lớn hơn độ dài
    ([5, 5, 5], 1, 1, 6),              # mọi đoạn
    ([5, 5, 5], 1, 2, 3),              # đoạn độ dài >= 2
    ([1, 2, 3], 3, 1, 1),              # cả mảng
    ([1, 2, 3], 4, 1, 0),              # k lớn hơn số giá trị phân biệt
    ([1, 1, 2, 2, 1, 1], 2, 2, None),
]
for nums, k, m, exp in cases:
    r = run_all(nums, k, m)
    assert len(set(r)) == 1, (nums, k, m, r)
    if exp is not None:
        assert r[0] == exp, (nums, k, m, r)
    print(nums, k, m, "->", r[0])

[7] 1 1 -> 1
[7] 1 2 -> 0
[5, 5, 5] 1 1 -> 6
[5, 5, 5] 1 2 -> 3
[1, 2, 3] 3 1 -> 1
[1, 2, 3] 4 1 -> 0
[1, 1, 2, 2, 1, 1] 2 2 -> 6


In [16]:
# 12.3 phản ví dụ: chỉ dùng F(k) (quên trừ F(k+1)) — đếm thừa các đoạn có nhiều hơn k giá trị
nums, k, m = [1, 1, 2, 2, 3, 3], 2, 2
print("chỉ F(k):", at_least(nums, k, k, m), "| đúng:", Solution().countSubarrays(nums, k, m), "| vét cạn:", brute(nums, k, m))

chỉ F(k): 5 | đúng: 2 | vét cạn: 2


In [17]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3859)
for _ in range(600):
    n = rng.randint(1, 12)
    nums = [rng.randint(1, 4) for _ in range(n)]
    k = rng.randint(1, 4); m = rng.randint(1, 4)
    r = run_all(nums, k, m)
    assert len(set(r)) == 1, (nums, k, m, r)
print("600 test nhỏ: OK")
for _ in range(60):
    n = rng.randint(100, 500)
    nums = [rng.randint(1, rng.randint(2, 8)) for _ in range(n)]
    k = rng.randint(1, 4); m = rng.randint(1, 6)
    r = run_all(nums, k, m, with_brute=False)
    assert len(set(r)) == 1
print("60 test vừa: OK")

600 test nhỏ: OK


60 test vừa: OK


In [18]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
nums = [rng.randint(1, 50) for _ in range(10**5)]
t0 = time.time(); a = Solution().countSubarrays(nums, 30, 3); t1 = time.time()
print("đáp án:", a, "| khớp một lượt:", a == one_pass(nums, 30, 3), "| thời gian < 2s:", t1 - t0 < 2)
same = [1] * 10**5
print("toàn một giá trị, k = m = 1:", Solution().countSubarrays(same, 1, 1), "= n(n+1)/2 =", 10**5 * (10**5 + 1) // 2)

đáp án: 0 | khớp một lượt: True | thời gian < 2s: True
toàn một giá trị, k = m = 1: 5000050000 = n(n+1)/2 = 5000050000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cửa sổ trượt trực tiếp cho "đúng `k`"

"Đúng `k` phân biệt" không đơn điệu ⇒ một con trỏ `l` không đủ; dễ bỏ sót đầu trái.

In [19]:
def wrong_direct(nums, k, m):
    cnt = Counter(); l = 0; ans = 0
    for r, x in enumerate(nums):
        cnt[x] += 1
        while len(cnt) > k:
            y = nums[l]; cnt[y] -= 1
            if cnt[y] == 0: del cnt[y]
            l += 1
        if len(cnt) == k and all(v >= m for v in cnt.values()):
            ans += 1                                 # SAI: chỉ đếm một đầu trái mỗi r
    return ans

print("sai:", wrong_direct([1, 1, 2], 2, 1), "| đúng:", Solution().countSubarrays([1, 1, 2], 2, 1))

sai: 1 | đúng: 2


**Vì sao sai:** với `r = 2` có hai đầu trái hợp lệ (`[1,1,2]` và `[1,2]`), cách trên chỉ đếm một. **Sửa:** `F(k) − F(k+1)`.

### Lỗi 2 — Quên xóa khóa khi tần suất về 0

`len(cnt)` khi đó vẫn đếm cả giá trị không còn trong cửa sổ ⇒ `d` bị thừa.

In [20]:
def wrong_no_delete(nums, lim, k, m):
    cnt = Counter(); t = 0; l = 0; ans = 0
    for x in nums:
        cnt[x] += 1
        if cnt[x] == m: t += 1
        while len(cnt) >= lim and t >= k:
            y = nums[l]; cnt[y] -= 1
            if cnt[y] == m - 1: t -= 1
            l += 1                                    # SAI: không xóa khóa có tần suất 0
        ans += l
    return ans

nums, k, m = [1, 2, 1, 2, 3, 3], 2, 1
print("sai:", wrong_no_delete(nums, 2, k, m) - wrong_no_delete(nums, 3, k, m),
      "| đúng:", Solution().countSubarrays(nums, k, m), "| vét cạn:", brute(nums, k, m))

sai: 6 | đúng: 8 | vét cạn: 8


### Lỗi 3 — Cập nhật `t` bằng `cnt[y] < m` (thay vì `== m - 1`)

`t` chỉ giảm **đúng một lần** khi tần suất vượt xuống dưới ngưỡng. Điều kiện `< m` làm giảm nhiều lần cho cùng một giá trị.

In [21]:
def wrong_t_update(nums, lim, k, m):
    cnt = Counter(); t = 0; l = 0; ans = 0
    for x in nums:
        cnt[x] += 1
        if cnt[x] == m: t += 1
        while len(cnt) >= lim and t >= k:
            y = nums[l]; cnt[y] -= 1
            if cnt[y] < m: t -= 1                    # SAI
            if cnt[y] == 0: del cnt[y]
            l += 1
        ans += l
    return ans

rng = random.Random(2)
for _ in range(5000):
    nums = [rng.randint(1, 3) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, 3); m = rng.randint(1, 3)
    w = wrong_t_update(nums, k, k, m) - wrong_t_update(nums, k + 1, k, m)
    if w != Solution().countSubarrays(nums, k, m):
        print("phản ví dụ:", nums, k, m, "| sai:", w, "| đúng:", Solution().countSubarrays(nums, k, m), "| vét cạn:", brute(nums, k, m))
        break

phản ví dụ: [3, 1, 3, 3] 1 2 | sai: 0 | đúng: 1 | vét cạn: 1


### Lỗi 4 — Điều kiện "`t == k`" trong vòng `while` của `F`

Dùng `t == k` thay vì `t >= k` làm mất tính đơn điệu (đoạn rộng hơn có thể có `t > k`), cửa sổ dừng sai chỗ.

In [22]:
def wrong_t_equal(nums, lim, k, m):
    cnt = Counter(); t = 0; l = 0; ans = 0
    for x in nums:
        cnt[x] += 1
        if cnt[x] == m: t += 1
        while len(cnt) >= lim and t == k:            # SAI: phải là t >= k
            y = nums[l]; cnt[y] -= 1
            if cnt[y] == m - 1: t -= 1
            if cnt[y] == 0: del cnt[y]
            l += 1
        ans += l
    return ans

rng = random.Random(1)
for _ in range(3000):
    nums = [rng.randint(1, 3) for _ in range(rng.randint(1, 9))]; k = rng.randint(1, 3); m = rng.randint(1, 3)
    w = wrong_t_equal(nums, k, k, m) - wrong_t_equal(nums, k + 1, k, m)
    if w != Solution().countSubarrays(nums, k, m):
        print("phản ví dụ:", nums, k, m, "| sai:", w, "| đúng:", Solution().countSubarrays(nums, k, m)); break

phản ví dụ: [3, 1, 2, 1] 1 1 | sai: 10 | đúng: 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đúng k"** trên đoạn con ⇒ thử **"ít nhất k − ít nhất k+1"** (hoặc "nhiều nhất k − nhiều nhất k−1").
- **Điều kiện "mỗi giá trị ≥ m lần"** ⇒ biến đếm `t` = số giá trị đạt ngưỡng, cập nhật khi tần suất **đi qua** ngưỡng.
- Cửa sổ trượt chỉ dùng được khi tính chất **đơn điệu** theo phép mở rộng/thu hẹp đoạn.
- Liên quan: LC 992 *Subarrays with K Different Integers*, LC 395 *Longest Substring with At Least K Repeating Characters*, LC 2537 *Count the Number of Good Subarrays*.

## 15. Biến thể của bài

1. **Không có điều kiện `m`** (`m = 1`): chính là LC 992.
2. **Nhiều nhất `k` giá trị phân biệt, mỗi giá trị ≥ `m` lần**: `Σ_{j=1..k}` của bài gốc với `j` thay `k`.
3. **Độ dài lớn nhất** thay vì số lượng: với mỗi `r`, đầu trái hợp lệ nhỏ nhất là `lo` (nếu `lo < hi`).
4. **Mỗi giá trị xuất hiện đúng `m` lần**: thêm ràng buộc trên — không còn đơn điệu, cần kỹ thuật khác (ví dụ hash theo tần suất).

In [23]:
# Biến thể 1: m = 1 trùng LC 992 (đúng k phân biệt)
def lc992(nums, k):
    def at_most(K):
        cnt = Counter(); l = ans = 0
        for r, x in enumerate(nums):
            cnt[x] += 1
            while len(cnt) > K:
                y = nums[l]; cnt[y] -= 1
                if cnt[y] == 0: del cnt[y]
                l += 1
            ans += r - l + 1
        return ans
    return at_most(k) - at_most(k - 1)

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(1, 12))]; k = rng.randint(1, 4)
    assert lc992(nums, k) == Solution().countSubarrays(nums, k, 1)
print("Biến thể m = 1 khớp LC 992.")

Biến thể m = 1 khớp LC 992.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | `Counter` mọi đoạn | `O(n³)` | `O(n)` | Kiểm thử |
| 2. Mở rộng từ `l` | Cập nhật dần, dừng sớm | `O(n²)` | `O(n)` | `n ≤ 5000` |
| 3. `F(k) − F(k+1)` | Hai tính chất đơn điệu | `O(n)` | `O(n)` | **Lời giải chuẩn** |
| 4. Một lượt | Hai con trỏ trái `lo`, `hi` | `O(n)` | `O(n)` | Khi muốn một lượt duyệt |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countSubarrays([2, 2, 2, 2], 1, 3)` = ?

**Bài 2 (tính tay).** `nums = [1, 1, 2, 2, 1]`, `k = 2`, `m = 2` — liệt kê các đoạn hợp lệ.

**Bài 3 (điền TODO).** Viết `longest_valid(nums, k, m)`: độ dài lớn nhất của đoạn hợp lệ (hoặc 0).

**Bài 4 (tự viết lại).** Viết `F(lim)` không dùng `Counter` mà dùng mảng `cnt` kích thước `max(nums) + 1` và biến `d` đếm số giá trị phân biệt.

**Bài 5 (mini challenge).** Đếm đoạn có **nhiều nhất** `k` giá trị phân biệt và mỗi giá trị ≥ `m` lần (Biến thể 2), kiểm chứng vét cạn.

In [24]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def longest_valid(nums, k, m):
    # TODO: chạy như one_pass; nếu lo < hi thì ứng viên độ dài r - lo + 1
    pass

In [25]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def at_most_k_valid(nums, k, m):
    # TODO: tổng countSubarrays(nums, j, m) với j = 1..k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `k = 1` nên đoạn chỉ chứa số 2. (2) Cần độ dài ≥ 3: đoạn dài 3 có 2 cái, dài 4 có 1 cái. (3) Đáp án `3`.

**Bài 2.** (1) Cần cả `1` và `2`, mỗi số ≥ 2 lần. (2) `[1,1,2,2]` (0..3), `[1,1,2,2,1]` (0..4). (3) `[1,2,2,1]` (1..4) cũng có hai số 1, hai số 2 ⇒ đáp án `3`.

**Bài 3.** (1) Với mỗi `r`, đầu trái hợp lệ là `[lo, hi)`. (2) Đoạn dài nhất kết thúc tại `r` bắt đầu ở `lo`. (3) Lấy max qua mọi `r`.

**Bài 4.** (1) `cnt = [0] * (max(nums) + 1)`. (2) `d += 1` khi `cnt[x]` từ 0 lên 1; `d -= 1` khi về 0. (3) Thay `len(cnt)` bằng `d`.

**Bài 5.** (1) Các tập "đúng `j`" rời nhau theo `j`. (2) Cộng `j = 1..k`. (3) `O(n·k)` — đủ cho kiểm thử.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().countSubarrays([2, 2, 2, 2], 1, 3) == 3
assert Solution().countSubarrays([1, 1, 2, 2, 1], 2, 2) == 3 == brute([1, 1, 2, 2, 1], 2, 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [27]:
# Bài 3
def longest_valid_sol(nums, k, m):
    ca = Counter(); lo = 0; cb = Counter(); tb = 0; hi = 0; best = 0
    for r, x in enumerate(nums):
        ca[x] += 1
        while len(ca) > k:
            y = nums[lo]; ca[y] -= 1
            if ca[y] == 0: del ca[y]
            lo += 1
        cb[x] += 1
        if cb[x] == m: tb += 1
        while len(cb) >= k and tb >= k:
            y = nums[hi]; cb[y] -= 1
            if cb[y] == m - 1: tb -= 1
            if cb[y] == 0: del cb[y]
            hi += 1
        if lo < hi:
            best = max(best, r - lo + 1)
    return best

def brute_longest(nums, k, m):
    n = len(nums); best = 0
    for l in range(n):
        for r in range(l, n):
            c = Counter(nums[l:r + 1])
            if len(c) == k and all(v >= m for v in c.values()): best = max(best, r - l + 1)
    return best

rng = random.Random(3)
for _ in range(400):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, 3); m = rng.randint(1, 3)
    assert longest_valid_sol(nums, k, m) == brute_longest(nums, k, m)
print("Bài 3 OK")

Bài 3 OK


In [28]:
# Bài 4
def F_array(nums, lim, k, m):
    cnt = [0] * (max(nums) + 1); d = t = l = ans = 0
    for x in nums:
        cnt[x] += 1
        if cnt[x] == 1: d += 1
        if cnt[x] == m: t += 1
        while d >= lim and t >= k:
            y = nums[l]; cnt[y] -= 1
            if cnt[y] == m - 1: t -= 1
            if cnt[y] == 0: d -= 1
            l += 1
        ans += l
    return ans

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(1, 5) for _ in range(rng.randint(1, 15))]; k = rng.randint(1, 4); m = rng.randint(1, 3)
    assert F_array(nums, k, k, m) - F_array(nums, k + 1, k, m) == Solution().countSubarrays(nums, k, m)
print("Bài 4 OK")

Bài 4 OK


In [29]:
# Bài 5
def at_most_k_valid_sol(nums, k, m):
    return sum(Solution().countSubarrays(nums, j, m) for j in range(1, k + 1))

def brute_at_most(nums, k, m):
    n = len(nums); c = 0
    for l in range(n):
        for r in range(l, n):
            cc = Counter(nums[l:r + 1])
            c += len(cc) <= k and all(v >= m for v in cc.values())
    return c

rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, 4); m = rng.randint(1, 3)
    assert at_most_k_valid_sol(nums, k, m) == brute_at_most(nums, k, m)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def F(lim):                       # #đoạn có d >= lim và t >= k
    cnt = Counter(); t = l = ans = 0
    for x in nums:
        cnt[x] += 1
        if cnt[x] == m: t += 1
        while len(cnt) >= lim and t >= k:
            y = nums[l]; cnt[y] -= 1
            if cnt[y] == m - 1: t -= 1
            if cnt[y] == 0: del cnt[y]
            l += 1
        ans += l
    return ans
return F(k) - F(k + 1)
```

**Checklist:**

- [ ] Dùng hiệu `F(k) − F(k+1)`, không trượt trực tiếp cho "đúng k"?
- [ ] Cập nhật `t` khi tần suất **đi qua** ngưỡng (`== m` khi tăng, `== m - 1` khi giảm)?
- [ ] Xóa khóa khi tần suất về 0?
- [ ] Điều kiện `while` dùng `t >= k` (đơn điệu)?
- [ ] Đáp án có thể ~`5·10^9` (kiểu 64-bit ở ngôn ngữ khác)?